***
# Homework 11: MapReduce using `PySpark`

**Course:** STAT 606 - Computing in Data Science and Statistics SP24

**Name:** Shrivats Sudhir

**NetID:** ssudhir2

**Email:** ssudhir2@wisc.edu

**Collaborators:** Samuel Merten, Amy Merkelz

**Date:** April 25th, 2024
***

## 1.) Preliminaries: Set up a Storage Bucket (2 points, spent $\approx$ 5 minutes)

**Before we get started, create a storage bucket for this project, just like you did for Homework 10, this time called `NetID-stat606s24-hw11`, where `NetID` is your Wisconsin `NetID` in all lower-case letters. All settings should be the same as the bucket you created for Homework 10.**

Completed the above procedure.

## 2.) Warmup: Interactive PySpark on GCP (3 points, spent $\approx$ 25 minutes)

**Before we can do anything in `PySpark`, we have to get a server up and running. Sign in to Google Cloud Platform, and make sure that you are in your project that you created in Homework 10. Recall that this project should be named `NetID-stat606s24`, where `NetID` is your Wisconsin `NetID` in all lower-case letters. Open Google Cloud Console and type:**

<h5 align="center"> gcloud dataproc clusters create CLUSTERNAME --region=REGION </h5>

**where `CLUSTERNAME` is the name you wish to give your cluster (e.g., stat606hw11 or something like that; you are free to name this however you like) and `REGION` is a valid region. You will need to wait a few minutes while Google Cloud sets up your cluster (i.e., gets some computers to serve as your nodes, installs necessary software on those computers, etc.). Once this process finishes, you will see a message to the effect of `Created [CLUSTERNAME] Cluster placed in zone [REGION]`. Once you have created this cluster, you should see it listed when you call**

<h5 align="center"> gcloud dataproc clusters list --region=REGION </h5>

**in the console, where `REGION` is the same as the argument supplied when you created the cluster.**

***Important warning:* any time you finish a working session (e.g., to take a break and come back again later), consider deleting your cluster with:**

<h5 align="center"> gcloud dataproc clusters delete CLUSTERNAME --region=REGION </h5>

**to ensure that you are not paying to leave a cluster sitting unused. Of course, when you come back to continue working, you will have to spin up the cluster again by following the instructions above. Bear in mind that any files that you create on the cluster are lost when you delete it, so be sure to move any files you want to keep into a storage bucket (we discuss this point at more length below).**

Completed the above procedure.

Given below is the terminal chunk that created the requested cluster:

```bash
ssudhir2@cloudshell:~ (ssudhir2-stat606s24)$ gcloud dataproc clusters create ssudhir2-stat606s24-hw11 --region=us-east1
Waiting on operation [projects/ssudhir2-stat606s24/regions/us-east1/operations/e1048b96-7afb-386b-b088-e6b870526a4a].
Waiting for cluster creation operation...                                                                                                                                  
WARNING: No image specified. Using the default image version. It is recommended to select a specific image version in production, as the default image version may change at any time.
WARNING: Failed to validate permissions required for default service account: '89254757867-compute@developer.gserviceaccount.com'. Cluster creation could still be successful if required permissions have been granted to the respective service accounts as mentioned in the document https://cloud.google.com/dataproc/docs/concepts/configuring-clusters/service-accounts#dataproc_service_accounts_2. This could be due to Cloud Resource Manager API hasn't been enabled in your project '89254757867' before or it is disabled. Enable it by visiting 'https://console.developers.google.com/apis/api/cloudresourcemanager.googleapis.com/overview?project=89254757867'.
WARNING: The firewall rules for specified network or subnetwork would allow ingress traffic from 0.0.0.0/0, which could be a security risk.
Waiting for cluster creation operation...done.                                                                                                                             
Created [https://dataproc.googleapis.com/v1/projects/ssudhir2-stat606s24/regions/us-east1/clusters/ssudhir2-stat606s24-hw11] Cluster placed in zone [us-east1-c].
```

**Okay, now that we have a cluster up and running, let’s try running an interactive `PySpark` session. To do that, we need to log onto our cluster. We will `ssh` to the master node on your Dataproc cluster. Double-check that your Dataproc cluster is up and running by calling**

<h5 align="center"> gcloud dataproc clusters list --region=REGION </h5>

**again (`REGION` should be set to whatever region you requested when you created the cluster).** 

```bash
ssudhir2@cloudshell:~ (ssudhir2-stat606s24)$ gcloud dataproc clusters list --region=us-east1
NAME: ssudhir2-stat606s24-hw11
PLATFORM: GCE
PRIMARY_WORKER_COUNT: 2
SECONDARY_WORKER_COUNT: 
STATUS: RUNNING
ZONE: us-east1-c
SCHEDULED_DELETE:
```

**If a cluster shows up in the list, go to the VM Instances dashboard, where you should see a few entries listed. These correspond to the nodes in your cluster. The names of these instances should all be prefixed with your cluster name. One of them should end with `-m`. This is the master node in your cluster. To `ssh` to it (i.e., log on to that machine), type the command**

<h5 align="center"> gcloud compute ssh MASTERNODE --project=PROJECT --zone=ZONE </h5>

**in the console, where `MASTERNODE` is the name of your cluster with the added suffix `-m` (something like `CLUSTERNAME-m`), `PROJECT` is the name of your project (something like `NetID-stat606s24`), and `ZONE` is the specific zone that your cluster is in. This will have a form like `REGION` or `REGION-X`, where `REGION` is your specific region specified when you launched the cluster, and `X` is a letter or number. If you’re not sure, you can find the zone of your cluster in the “Zone” column of the VM instances dashboard.**

**Note: you may be prompted to create an RSA key pair when logging on to your master node. Go ahead and create a password for this if you wish, or feel free to use the “no password” option, since we won’t be working with any sensitive files in this exercise. Of course, when working in an actual production environment, you should be careful to follow good security practices such as using secure passwords, encryption, etc.**

**If all goes well, it won’t look like much has changed, except you’ll see that your prompt in the console has changed to something like `NetID@CLUSTERNAME-m`. Alternatively, you can type the command hostname in the console, which should produce an output of the form `CLUSTERNAME-m`.**

Currently at the SSH-in-browser bash shell terminal:

```bash
Linux ssudhir2-stat606s24-hw11-m 5.10.0-0.deb10.16-cloud-amd64 #1 SMP Debian 5.10.127-2~bpo10+1 (2022-07-28) x86_64

The programs included with the Debian GNU/Linux system are free software;
the exact distribution terms for each program are described in the
individual files in /usr/share/doc/*/copyright.

Debian GNU/Linux comes with ABSOLUTELY NO WARRANTY, to the extent
permitted by applicable law.
ssudhir2@ssudhir2-stat606s24-hw11-m:~$ 
```

**Now you can start an interactive PySpark session by typing `pyspark` in the console.**


**When you do this, you’ll see some text appear, giving some setup information and information about the version of Spark, and then you’ll see the interactive prompt (>>>).** 

**The `numbers.txt` file from lecture is available at:**

<h5 align="center"> gs://uw-stat606s24-hw11/numbers.txt </h5>

**Read it into an RDD in your PySpark interactive session and use a sequence of RDD transformations and RDD actions to compute how many of the numbers in the file are prime.** 

**You may make use of the function is_prime, which is defined in the Python file:**

<h5 align="center"> gs://uw-stat606s24-hw11/prime.py </h5>

**Save the answer in a variable called number_of_primes in your Jupyter notebook file for submission. Note: you can quit an interactive PySpark session either by typing `quit()` at the prompt or by typing ctrl-D.3**

**Please also copy-paste into your Jupyter notebook file the sequence of PySpark commands that you ran to obtain this answer. Important: paste these into a Raw NBConvert or Markdown cell, not a Code cell. If you paste these commands into a Jupyter Code cell, the grader script will try to run your PySpark commands in plain old Python, which will cause errors.**

**Reminder: if you aren’t going to continue working on the next problem immediately, save GCP credits by deleting your cluster.**

First, we go to the `CLUSTERNAME-m` bash command terminal to check all files that are located in `gs://uw-stat606s24-hw11`:

```bash
ssudhir2@ssudhir2-stat606s24-hw11-m:~$ gsutil ls gs://uw-stat606s24-hw11/
gs://uw-stat606s24-hw11/NOAA_MSN_temps.csv
gs://uw-stat606s24-hw11/numbers.txt
gs://uw-stat606s24-hw11/prime.py
gs://uw-stat606s24-hw11/ps_wordcount.py
gs://uw-stat606s24-hw11/war_and_peace.txt
```

We can see that both our `numbers.txt` as well as `prime.py` are located in the cloud directory.

We now use `cat` to check the contents of the file as follows:

```bash
ssudhir2@ssudhir2-stat606s24-hw11-m:~$ gsutil cat gs://uw-stat606s24-hw11/prime.py
import math

def is_prime(n):
    if n <= 1: # Primes must be naturals; 1 is not prime.
        return False
    for x in range(2,int(math.sqrt(n))+1):
        if n%x==0:
            return False
    return True 
```

Now, we load into Pyspark as follows:

```bash
ssudhir2@ssudhir2-stat606s24-hw11-m:~$ pyspark
Python 3.8.15 | packaged by conda-forge | (default, Nov 22 2022, 08:46:39) 
[GCC 10.4.0] on linux
Type "help", "copyright", "credits" or "license" for more information.
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/04/28 04:43:26 INFO org.apache.spark.SparkEnv: Registering MapOutputTracker
24/04/28 04:43:26 INFO org.apache.spark.SparkEnv: Registering BlockManagerMaster
24/04/28 04:43:26 INFO org.apache.spark.SparkEnv: Registering BlockManagerMasterHeartbeat
24/04/28 04:43:27 INFO org.apache.spark.SparkEnv: Registering OutputCommitCoordinator
Welcome to
      ____              __
     / __/__  ___ _____/ /__
    _\ \/ _ \/ _ `/ __/  '_/
   /__ / .__/\_,_/_/ /_/\_\   version 3.1.3
      /_/

Using Python version 3.8.15 (default, Nov 22 2022 08:46:39)
Spark context Web UI available at http://ssudhir2-stat606s24-hw11-m.us-east1-c.c.ssudhir2-stat606s24.internal:39851
Spark context available as 'sc' (master = yarn, app id = application_1714278108368_0002).
SparkSession available as 'spark'.
>>>
```

We first load the `numbers.txt` data into pyspark using `sc.textfile()` and `collect()`, then we convert the list of string into a list of integers as follows:

```bash
>>> data = sc.textFile('gs://uw-stat606s24-hw11/numbers.txt')
>>> data.collect()
['10', '23', '16', '7', '12', '0', '1', '1', '2', '3', '5', '8', '-1', '42', '64', '101', '-101', '3']
>>> numbers = [int(i) for i in data.collect()]
>>> print(numbers)
[10, 23, 16, 7, 12, 0, 1, 1, 2, 3, 5, 8, -1, 42, 64, 101, -101, 3]
```

We now load the `prime.py` file into pyspark using `sc.addPyfile()` and the relevant import statements as follows:

```bash
>>> sc.addPyFile('gs://uw-stat606s24-hw11/prime.py')
>>> from prime import *
```

Now, we run the `is_prime()` function from `prime.py` on the list `numbers`, and add up all `True` values to get our answer:

```bash
>>> prime_results = [is_prime(num) for num in numbers]
>>> prime_results
[False, True, False, True, False, False, False, False, True, True, True, False, False, False, False, True, False, True]
>>> number_of_primes = sum(prime_results)
>>> print(number_of_primes)
7
>>> quit()
```

Deleting dataproc cluster:

```bash
ssudhir2@cloudshell:~ (ssudhir2-stat606s24)$ gcloud dataproc clusters delete ssudhir2-stat606s24-hw11 --region=us-east1
The cluster 'ssudhir2-stat606s24-hw11' and all attached disks will be deleted.

Do you want to continue (Y/n)?  Y

Waiting on operation [projects/ssudhir2-stat606s24/regions/us-east1/operations/701448dd-53c5-327c-9933-0f954affa1db].
Waiting for cluster deletion operation...done.          
Deleted [https://dataproc.googleapis.com/v1/projects/ssudhir2-stat606s24/regions/us-east1/clusters/ssudhir2-stat606s24-hw11].
```

In [1]:
number_of_primes = 7

## 3.) Submitting a Job to Spark (6 points, spent $\approx$ 15 minutes)

**Now let’s try writing a PySpark script and submitting it to your Dataproc server.**

**First things first: make sure that you have a Dataproc cluster up and running by typing**

<h5 align="center"> gcloud dataproc clusters list --region=REGION </h5>

**where `REGION` is the region you specified upon cluster creation. Alternatively, you can pull up the VM instances dashboard to see a list of your currently-running VM instances (this list will include any running Dataproc clusters).** 

**If you don’t have a Dataproc cluster up and running, follow the instructions from the previous problem to create one.**

Re-creating Dataproc cluster:

```bash
ssudhir2@cloudshell:~ (ssudhir2-stat606s24)$ gcloud dataproc clusters create ssudhir2-stat606s24-hw11 --region=us-east1
Waiting on operation [projects/ssudhir2-stat606s24/regions/us-east1/operations/022548de-673a-3f10-bfd1-2184400019d4].
Waiting for cluster creation operation...                                        
WARNING: No image specified. Using the default image version. It is recommended to select a specific image version in production, as the default image version may change at any time.
WARNING: Failed to validate permissions required for default service account: '89254757867-compute@developer.gserviceaccount.com'. Cluster creation could still be successful if required permissions have been granted to the respective service accounts as mentioned in the document https://cloud.google.com/dataproc/docs/concepts/configuring-clusters/service-accounts#dataproc_service_accounts_2. This could be due to Cloud Resource Manager API hasn't been enabled in your project '89254757867' before or it is disabled. Enable it by visiting 'https://console.developers.google.com/apis/api/cloudresourcemanager.googleapis.com/overview?project=89254757867'.
WARNING: The firewall rules for specified network or subnetwork would allow ingress traffic from 0.0.0.0/0, which could be a security risk.
WARNING: The specified custom staging bucket 'dataproc-staging-us-east1-89254757867-y0f1tueq' is not using uniform bucket level access IAM configuration. It is recommended to update bucket to enable the same. See https://cloud.google.com/storage/docs/uniform-bucket-level-access.
Waiting for cluster creation operation...done.
```

Checking for clusters:

```bash
ssudhir2@cloudshell:~ (ssudhir2-stat606s24)$ gcloud dataproc clusters list --region=us-east1
NAME: ssudhir2-stat606s24-hw11
PLATFORM: GCE
PRIMARY_WORKER_COUNT: 2
SECONDARY_WORKER_COUNT: 
STATUS: RUNNING
ZONE: us-east1-d
SCHEDULED_DELETE: 
```

**Now let’s try running our example from lecture. The `ps_wordcount.py` script from the lecture slides is available at:**

<h5 align="center"> gs://uw-stat606s24-hw11/ps_wordcount.py </h5>

**(alternatively, you can download the demo code from this week’s lecture and upload a copy to your own storage bucket).** 

**The file at**

<h5 align="center"> gs://uw-stat606s24-hw11/war_and_peace.txt </h5>

**contains a slightly modified version of the Project Gutenberg UTF-8 copy of *Leo Tolstoy’s War and Peace* Submit a PySpark job to your Dataproc server that runs `ps_wordcount.py` on `war_and_peace.txt` and outputs the results to a directory:**

<h5 align="center"> gs://NetID-stat606s24-hw11/WP_wordcount </h5>

**where once again `NetID` is your `NetID` in all lower-case. Please also copy-paste the command that you called to launch this job into a Raw NBConvert cell or a Markdown cell in your Jupyter notebook file.**

We first use `cat` to see what the `ps_wordcount.py` file looks like:

```bash
ssudhir2@ssudhir2-stat606s24-hw11-m:~$ gsutil cat gs://uw-stat606s24-hw11/ps_wordcount.py
from pyspark import SparkConf, SparkContext
import sys

# This script takes two arguments, an input file and output directory.
if len(sys.argv) != 3:
    print('Usage: ' + sys.argv[0] + ' <in> <out>')
    sys.exit(1)
inputlocation = sys.argv[1]
outputlocation = sys.argv[2]

# Set up the configuration and job context
conf = SparkConf().setAppName('WordCount')
sc = SparkContext(conf=conf)

# Read in the dataset and immediately transform all the lines into arrays.
data = sc.textFile(inputlocation)
data_flat = data.flatMap(lambda line: line.split())
wordkeys = data_flat.map(lambda w: (w.lower(),1) )
wordcounts = wordkeys.reduceByKey(lambda x,y: x+y)

# Save the results in the specified output directory.
wordcounts.saveAsTextFile(outputlocation)
sc.stop() # Let Spark know that the job is done.
```

We can see that the script takes two command line arguments, input file and output directory.

Thus, with this knowledge, we now run `ps_wordcount.py` on the bash command terminal as follows:

```bash
ssudhir2@ssudhir2-stat606s24-hw11-m:~$ spark-submit gs://uw-stat606s24-hw11/ps_wordcount.py gs://uw-stat606s24-hw11/war_and_peace.txt gs://ssudhir2-stat606s24-hw11/WP_wordcount
24/04/28 17:29:45 INFO org.apache.spark.SparkEnv: Registering MapOutputTracker
24/04/28 17:29:45 INFO org.apache.spark.SparkEnv: Registering BlockManagerMaster
24/04/28 17:29:45 INFO org.apache.spark.SparkEnv: Registering BlockManagerMasterHeartbeat
24/04/28 17:29:45 INFO org.apache.spark.SparkEnv: Registering OutputCommitCoordinator
24/04/28 17:29:45 INFO org.sparkproject.jetty.util.log: Logging initialized @6010ms to org.sparkproject.jetty.util.log.Slf4jLog
24/04/28 17:29:45 INFO org.sparkproject.jetty.server.Server: jetty-9.4.40.v20210413; built: 2021-04-13T20:42:42.668Z; git: b881a572662e1943a14ae12e7e1207989f218b74; jvm 1.8.0_402-b06
24/04/28 17:29:45 INFO org.sparkproject.jetty.server.Server: Started @6154ms
24/04/28 17:29:45 INFO org.sparkproject.jetty.server.AbstractConnector: Started ServerConnector@37a9c617{HTTP/1.1, (http/1.1)}{0.0.0.0:42077}
24/04/28 17:29:46 INFO org.apache.hadoop.yarn.client.RMProxy: Connecting to ResourceManager at ssudhir2-stat606s24-hw11-m/10.142.0.9:8032
24/04/28 17:29:46 INFO org.apache.hadoop.yarn.client.AHSProxy: Connecting to Application History server at ssudhir2-stat606s24-hw11-m/10.142.0.9:10200
24/04/28 17:29:47 INFO org.apache.hadoop.conf.Configuration: resource-types.xml not found
24/04/28 17:29:47 INFO org.apache.hadoop.yarn.util.resource.ResourceUtils: Unable to find 'resource-types.xml'.
24/04/28 17:29:48 INFO org.apache.hadoop.yarn.client.api.impl.YarnClientImpl: Submitted application application_1714322621889_0003
24/04/28 17:29:49 INFO org.apache.hadoop.yarn.client.RMProxy: Connecting to ResourceManager at ssudhir2-stat606s24-hw11-m/10.142.0.9:8030
24/04/28 17:29:50 INFO com.google.cloud.hadoop.repackaged.gcs.com.google.cloud.hadoop.gcsio.GoogleCloudStorageImpl: Ignoring exception of type GoogleJsonResponseException; verified object already exists with desired state.
24/04/28 17:29:51 INFO org.apache.hadoop.mapred.FileInputFormat: Total input files to process : 1
24/04/28 17:29:52 INFO com.google.cloud.hadoop.fs.gcs.GhfsStorageStatistics: Detected potential high latency for operation op_mkdirs. latencyMs=236; previousMaxLatencyMs=150; operationCount=2; context=gs://ssudhir2-stat606s24-hw11/WP_wordcount/_temporary/0
24/04/28 17:30:01 INFO com.google.cloud.hadoop.repackaged.gcs.com.google.cloud.hadoop.gcsio.GoogleCloudStorageFileSystem: Successfully repaired 'gs://ssudhir2-stat606s24-hw11/WP_wordcount/' directory.
24/04/28 17:30:01 INFO com.google.cloud.hadoop.fs.gcs.GhfsStorageStatistics: Detected potential high latency for operation op_delete. latencyMs=449; previousMaxLatencyMs=0; operationCount=1; context=gs://ssudhir2-stat606s24-hw11/WP_wordcount/_temporary
24/04/28 17:30:01 INFO com.google.cloud.hadoop.fs.gcs.GhfsStorageStatistics: Detected potential high latency for operation stream_write_close_operations. latencyMs=186; previousMaxLatencyMs=0; operationCount=1; context=gs://ssudhir2-stat606s24-hw11/WP_wordcount/_SUCCESS
24/04/28 17:30:01 INFO org.sparkproject.jetty.server.AbstractConnector: Stopped Spark@37a9c617{HTTP/1.1, (http/1.1)}{0.0.0.0:0}
24/04/28 17:30:02 INFO com.google.cloud.hadoop.fs.gcs.GhfsStorageStatistics: Detected potential high latency for operation op_rename. latencyMs=173; previousMaxLatencyMs=0; operationCount=1; context=rename(gs://dataproc-temp-us-east1-89254757867-nbcdhrwu/7e2aeb46-f6dc-46f1-988c-63c7fa7cb921/spark-job-history/application_1714322621889_0003.inprogress -> gs://dataproc-temp-us-east1-89254757867-nbcdhrwu/7e2aeb46-f6dc-46f1-988c-63c7fa7cb921/spark-job-history/application_1714322621889_0003)
```

Now, we check if the job was run successfully on our created cluster:

```bash
ssudhir2@ssudhir2-stat606s24-hw11-m:~$ gsutil ls gs://ssudhir2-stat606s24-hw11/WP_wordcount/
gs://ssudhir2-stat606s24-hw11/WP_wordcount/
gs://ssudhir2-stat606s24-hw11/WP_wordcount/_SUCCESS
gs://ssudhir2-stat606s24-hw11/WP_wordcount/part-00000
gs://ssudhir2-stat606s24-hw11/WP_wordcount/part-00001
```

**Concatenate the output of your script and store it in a file in your storage bucket at:**

<h5 align="center"> gs://NetID-stat606s24-hw11/wp_output.txt. </h5>

**please also include a copy of this file in your submission.**

Concatenating `part-00000` and `part-00001` in our bash terminal as follows:

```bash
ssudhir2@ssudhir2-stat606s24-hw11-m:~$ gsutil cat gs://ssudhir2-stat606s24-hw11/WP_wordcount/part-* > wp_output.txt
ssudhir2@ssudhir2-stat606s24-hw11-m:~$ ls
wp_output.txt
```

Now, we transfer `wp_output.txt` to the cloud bucket `gs://ssudhir2-stat606s24-hw11/` as follows:

```bash
ssudhir2@ssudhir2-stat606s24-hw11-m:~$ gsutil cp wp_output.txt gs://ssudhir2-stat606s24-hw11/
Copying file://wp_output.txt [Content-Type=text/plain]...
/ [0 files][    0.0 B/648.9 KiB]                                             / [1 files][648.9 KiB/648.9 KiB]                                                
Operation completed over 1 objects/648.9 KiB.                                    
```

Now, we check if our output is in the cloud directory as expected:

```bash
ssudhir2@ssudhir2-stat606s24-hw11-m:~$ gsutil ls gs://ssudhir2-stat606s24-hw11/
gs://ssudhir2-stat606s24-hw11/wp_output.txt
gs://ssudhir2-stat606s24-hw11/WP_wordcount/
```

## 4.) Climate Data Revisited (9 points, spent $\approx$ 50 minutes)

**I used NOAA’s Climate Data Online service to collect daily historical temperature data for Madison, WI, which has been gathered at Dane County Airport since 1939.** 

**I have made this data available on GCP at:**

<h5 align="center"> gs://uw-stat606s24-hw11/NOAA_MSN_temps.csv </h5>

**Each line of this file has the form: `DATE,TMAX,TMIN`**

**where `TMAX` and `TMIN` are integers describing the maximum and minimum temperatures (in degrees Fahrenheit) on a given day, and `DATE` encodes a date in the form `YYYYMMDD`.**

**You can see a few lines of the file by writing something like**

<h5 align="center"> gsutil cat -r 0-89 gs://uw-stat606s24-hw11/NOAA_MSN_temps.csv </h5>

```bash
ssudhir2@ssudhir2-stat606s24-hw11-m:~$ gsutil cat -r 0-89 gs://uw-stat606s24-hw11/NOAA_MSN_temps.csv
19391001,67,33
19391002,70,38
19391003,74,48
19391004,81,51
19391005,70,56
19391006,77,45
```

**to print out the first bytes of the file (six lines, at 15 bytes per line including the trailing new lines). This `-r` flag to the `gsutil cat` command is the closest thing (to the best of my knowledge, anyway) that `gsutil` has to the UNIX `head` command.** 

**Important: be careful when performing read operations like this with very large files. Reading multiple GBs or, worse, TBs of text into less or a similar command-line program can be very slow!**

**Write a PySpark script that reads two arguments from the command line, corresponding to an input file and an output directory, in that order (the same as the arguments for `ps_wordcount.py`) and computes the average maximum and minimum temperature for every year in the data set. The output should be of the form: `YYYY, avgmax, avgmin`**

**where `YYYY` is an integer encoding a year, and `avgmax` and `avgmin` are floats encoding the average maximum and minimum temperatures, respectively, for that year.**

**Note: the precise formatting here does not matter—just make sure that your output has a line for each year in the data set and the maximum and minimum temperature are ordered correctly. So, for example, an output like `YYYY, (avgmax, avgmin)` is also fine.** 

**Save your script in a file called `ps_year_avgs.py` and include copies in both your storage bucket and your submission. If you wrote any additional Python code (e.g., function definitions in a separate Python file), please also include this in your submission.** 

**Hint: you may find the `reduceByKey` and `mapValues` transformations to be especially useful**

Completed the above procedure. Submitted as `ps_year_avgs.py` in both canvas submission, as well as cloud bucket `ssudhir2-stat606s24-hw11`.

```bash
ssudhir2@ssudhir2-stat606s24-hw11-m:~$ gsutil cp ps_year_avgs.py gs://ssudhir2-stat606s24-hw11/
Copying file://ps_year_avgs.py [Content-Type=text/x-python]...
/ [1 files][  3.0 KiB/  3.0 KiB]                                                
Operation completed over 1 objects/3.0 KiB.                
```

**Run `ps_year_avgs.py` on the file `NOAA_MSN_temps.csv` in PySpark on a GCP Dataproc server.** 

**Concatenate the output of your job into a single file called `avgs.txt` and save this file in your storage bucket for this homework, and please also include a copy in your submission.**

First, we check the contents of our cloud bucket `ssudhir2-stat606s24-hw11`:

```bash
ssudhir2@ssudhir2-stat606s24-hw11-m:~$ gsutil ls gs://ssudhir2-stat606s24-hw11/
gs://ssudhir2-stat606s24-hw11/ps_year_avgs.py
gs://ssudhir2-stat606s24-hw11/wp_output.txt
gs://ssudhir2-stat606s24-hw11/WP_wordcount/
```

We can see that our python script `ps_year_avgs.py` exists in our cloud bucket.

Now, we can run the script using `spark-submit` as follows:

```bash
ssudhir2@ssudhir2-stat606s24-hw11-m:~$ spark-submit gs://ssudhir2-stat606s24-hw11/ps_year_avgs.py gs://uw-stat606s24-hw11/NOAA_MSN_temps.csv gs://ssudhir2-stat606s24-hw11/WP_averages/
24/04/28 23:37:03 INFO org.apache.spark.SparkEnv: Registering MapOutputTracker
24/04/28 23:37:03 INFO org.apache.spark.SparkEnv: Registering BlockManagerMaster
24/04/28 23:37:04 INFO org.apache.spark.SparkEnv: Registering BlockManagerMasterHeartbeat
24/04/28 23:37:04 INFO org.apache.spark.SparkEnv: Registering OutputCommitCoordinator
24/04/28 23:37:04 INFO org.sparkproject.jetty.util.log: Logging initialized @5658ms to org.sparkproject.jetty.util.log.Slf4jLog
24/04/28 23:37:04 INFO org.sparkproject.jetty.server.Server: jetty-9.4.40.v20210413; built: 2021-04-13T20:42:42.668Z; git: b881a572662e1943a14ae12e7e1207989f218b74; jvm 1.8.0_402-b06
24/04/28 23:37:04 INFO org.sparkproject.jetty.server.Server: Started @5803ms
24/04/28 23:37:04 INFO org.sparkproject.jetty.server.AbstractConnector: Started ServerConnector@68a55f8a{HTTP/1.1, (http/1.1)}{0.0.0.0:41577}
24/04/28 23:37:05 INFO org.apache.hadoop.yarn.client.RMProxy: Connecting to ResourceManager at ssudhir2-stat606s24-hw11-m/10.142.0.12:8032
24/04/28 23:37:05 INFO org.apache.hadoop.yarn.client.AHSProxy: Connecting to Application History server at ssudhir2-stat606s24-hw11-m/10.142.0.12:10200
24/04/28 23:37:05 INFO org.apache.hadoop.conf.Configuration: resource-types.xml not found
24/04/28 23:37:05 INFO org.apache.hadoop.yarn.util.resource.ResourceUtils: Unable to find 'resource-types.xml'.
24/04/28 23:37:06 INFO org.apache.hadoop.yarn.client.api.impl.YarnClientImpl: Submitted application application_1714341781891_0008
24/04/28 23:37:07 INFO org.apache.hadoop.yarn.client.RMProxy: Connecting to ResourceManager at ssudhir2-stat606s24-hw11-m/10.142.0.12:8030
24/04/28 23:37:08 INFO com.google.cloud.hadoop.repackaged.gcs.com.google.cloud.hadoop.gcsio.GoogleCloudStorageImpl: Ignoring exception of type GoogleJsonResponseException; verified object already exists with desired state.
24/04/28 23:37:10 INFO org.apache.hadoop.mapred.FileInputFormat: Total input files to process : 1
24/04/28 23:37:16 INFO com.google.cloud.hadoop.fs.gcs.GhfsStorageStatistics: Detected potential high latency for operation op_mkdirs. latencyMs=190; previousMaxLatencyMs=130; operationCount=2; context=gs://ssudhir2-stat606s24-hw11/WP_averages/_temporary/0
24/04/28 23:37:21 INFO com.google.cloud.hadoop.repackaged.gcs.com.google.cloud.hadoop.gcsio.GoogleCloudStorageFileSystem: Successfully repaired 'gs://ssudhir2-stat606s24-hw11/WP_averages/' directory.
24/04/28 23:37:21 INFO com.google.cloud.hadoop.fs.gcs.GhfsStorageStatistics: Detected potential high latency for operation op_delete. latencyMs=420; previousMaxLatencyMs=0; operationCount=1; context=gs://ssudhir2-stat606s24-hw11/WP_averages/_temporary
24/04/28 23:37:21 INFO com.google.cloud.hadoop.fs.gcs.GhfsStorageStatistics: Detected potential high latency for operation stream_write_close_operations. latencyMs=191; previousMaxLatencyMs=0; operationCount=1; context=gs://ssudhir2-stat606s24-hw11/WP_averages/_SUCCESS
24/04/28 23:37:21 INFO org.sparkproject.jetty.server.AbstractConnector: Stopped Spark@68a55f8a{HTTP/1.1, (http/1.1)}{0.0.0.0:0}
24/04/28 23:37:22 INFO com.google.cloud.hadoop.fs.gcs.GhfsStorageStatistics: Detected potential high latency for operation op_rename. latencyMs=159; previousMaxLatencyMs=0; operationCount=1; context=rename(gs://dataproc-temp-us-east1-89254757867-nbcdhrwu/e4eba3e9-a7d5-490f-86ed-ed80b0f70b03/spark-job-history/application_1714341781891_0008.inprogress -> gs://dataproc-temp-us-east1-89254757867-nbcdhrwu/e4eba3e9-a7d5-490f-86ed-ed80b0f70b03/spark-job-history/application_1714341781891_0008)
```

Now, we check if the job was run successfully on our created cluster:

```bash
ssudhir2@ssudhir2-stat606s24-hw11-m:~$ gsutil ls  gs://ssudhir2-stat606s24-hw11/WP_averages/
gs://ssudhir2-stat606s24-hw11/WP_averages/
gs://ssudhir2-stat606s24-hw11/WP_averages/_SUCCESS
gs://ssudhir2-stat606s24-hw11/WP_averages/part-00000
gs://ssudhir2-stat606s24-hw11/WP_averages/part-00001
gs://ssudhir2-stat606s24-hw11/WP_averages/part-00002
gs://ssudhir2-stat606s24-hw11/WP_averages/part-00003
```

Concatenating `part-00000`, `part-00001`, `part-00002`, and `part-00003` in our bash terminal as follows:

```bash
ssudhir2@ssudhir2-stat606s24-hw11-m:~$ gsutil cat gs://ssudhir2-stat606s24-hw11/WP_averages/part-* > avgs.txt
ssudhir2@ssudhir2-stat606s24-hw11-m:~$ ls
avgs.txt  ps_year_avgs.py  wp_output.txt
```

Now, we transfer `avgs.txt` to the cloud bucket `gs://ssudhir2-stat606s24-hw11/` as follows:

```bash
ssudhir2@ssudhir2-stat606s24-hw11-m:~$ gsutil cp avgs.txt gs://ssudhir2-stat606s24-hw11/
Copying file://avgs.txt [Content-Type=text/plain]...
/ [1 files][  1.7 KiB/  1.7 KiB]                                                
Operation completed over 1 objects/1.7 KiB.                                        
```

Now, we check if our output is in the cloud directory as expected:

```bash
ssudhir2@ssudhir2-stat606s24-hw11-m:~$ gsutil ls gs://ssudhir2-stat606s24-hw11/
gs://ssudhir2-stat606s24-hw11/avgs.txt
gs://ssudhir2-stat606s24-hw11/ps_year_avgs.py
gs://ssudhir2-stat606s24-hw11/wp_output.txt
gs://ssudhir2-stat606s24-hw11/WP_averages/
gs://ssudhir2-stat606s24-hw11/WP_wordcount/
```

**Write a PySpark script whose command line arguments are the same as those of `ps_wordcount.py` and `ps_year_avgs.py` and that computes, for each year in the data set, the day on which the maximum temperature was achieved and the day on which the minimum temperature was achieved (you may break ties as you see fit).**

**That is, each row of the output should be of a form like: `YYYY, MM-DD, mm-dd` where `MM-DD` encodes the month and day on which the maximum occurred and `mm-dd` encodes the month and day on which the minimum occurred.** 

**Note: the precise formatting here does not matter—just make sure that your output has a line for each year in the data set and the maximum and minimum temperature days are ordered correctly and in the correct `MM-DD` format, that is fine. So, for example, an output like: `YYYY, (MM-DD, mm-dd)` or `YYYY, 'MM-DD' 'mm-dd'` is also fine.** 

**Save your script in a file called `ps_year_extremes.py`. Please include a copy of this script in your storage bucket and include a copy in your submission.**

**If you wrote any additional Python code (e.g., function definitions in a separate Python file), please also include this in your submission.** 

**Hint: you may find it easiest to find the maximum and minimum separately, and then combine the two RDDs using the RDD transformation join.**

Completed the above procedure. Submitted as `ps_year_extremes.py` in both canvas submission, as well as cloud bucket `ssudhir2-stat606s24-hw11`.

```bash
ssudhir2@ssudhir2-stat606s24-hw11-m:~$ gsutil cp ps_year_extremes.py gs://ssudhir2-stat606s24-hw11/
Copying file://ps_year_extremes.py [Content-Type=text/x-python]...
/ [1 files][  5.5 KiB/  5.5 KiB]                                                
Operation completed over 1 objects/5.5 KiB.                                      
```

**Run your `ps_year_extremes.py` script on the file `NOAA_MSN_temps.csv` in PySpark on a GCP Dataproc server.** 

**Concatenate the output of your job into a single file called `extremes.txt` and save this file in your storage bucket for this homework, and please also include a copy in your submission.**

First, we check the contents of our cloud bucket `ssudhir2-stat606s24-hw11`:

```bash
ssudhir2@ssudhir2-stat606s24-hw11-m:~$ gsutil ls gs://ssudhir2-stat606s24-hw11/
gs://ssudhir2-stat606s24-hw11/avgs.txt
gs://ssudhir2-stat606s24-hw11/ps_year_avgs.py
gs://ssudhir2-stat606s24-hw11/ps_year_extremes.py
gs://ssudhir2-stat606s24-hw11/wp_output.txt
gs://ssudhir2-stat606s24-hw11/WP_averages/
gs://ssudhir2-stat606s24-hw11/WP_wordcount/
```

We can see that our python script `ps_year_extremes.py` exists in our cloud bucket.

Now, we can run the script using `spark-submit` as follows:

```bash
ssudhir2@ssudhir2-stat606s24-hw11-m:~$ spark-submit gs://ssudhir2-stat606s24-hw11/ps_year_extremes.py gs://uw-stat606s24-hw11/NOAA_MSN_temps.csv gs://ssudhir2-stat606s24-hw11/WP_extremes/
24/04/29 00:54:55 INFO org.apache.spark.SparkEnv: Registering MapOutputTracker
24/04/29 00:54:55 INFO org.apache.spark.SparkEnv: Registering BlockManagerMaster
24/04/29 00:54:56 INFO org.apache.spark.SparkEnv: Registering BlockManagerMasterHeartbeat
24/04/29 00:54:56 INFO org.apache.spark.SparkEnv: Registering OutputCommitCoordinator
24/04/29 00:54:56 INFO org.sparkproject.jetty.util.log: Logging initialized @5937ms to org.sparkproject.jetty.util.log.Slf4jLog
24/04/29 00:54:56 INFO org.sparkproject.jetty.server.Server: jetty-9.4.40.v20210413; built: 2021-04-13T20:42:42.668Z; git: b881a572662e1943a14ae12e7e1207989f218b74; jvm 1.8.0_402-b06
24/04/29 00:54:56 INFO org.sparkproject.jetty.server.Server: Started @6062ms
24/04/29 00:54:56 INFO org.sparkproject.jetty.server.AbstractConnector: Started ServerConnector@3b14e2b3{HTTP/1.1, (http/1.1)}{0.0.0.0:38087}
24/04/29 00:54:57 INFO org.apache.hadoop.yarn.client.RMProxy: Connecting to ResourceManager at ssudhir2-stat606s24-hw11-m/10.142.0.12:8032
24/04/29 00:54:57 INFO org.apache.hadoop.yarn.client.AHSProxy: Connecting to Application History server at ssudhir2-stat606s24-hw11-m/10.142.0.12:10200
24/04/29 00:54:58 INFO org.apache.hadoop.conf.Configuration: resource-types.xml not found
24/04/29 00:54:58 INFO org.apache.hadoop.yarn.util.resource.ResourceUtils: Unable to find 'resource-types.xml'.
24/04/29 00:54:58 INFO org.apache.hadoop.yarn.client.api.impl.YarnClientImpl: Submitted application application_1714341781891_0009
24/04/29 00:54:59 INFO org.apache.hadoop.yarn.client.RMProxy: Connecting to ResourceManager at ssudhir2-stat606s24-hw11-m/10.142.0.12:8030
24/04/29 00:55:01 INFO com.google.cloud.hadoop.repackaged.gcs.com.google.cloud.hadoop.gcsio.GoogleCloudStorageImpl: Ignoring exception of type GoogleJsonResponseException; verified object already exists with desired state.
24/04/29 00:55:01 INFO com.google.cloud.hadoop.fs.gcs.GhfsStorageStatistics: Detected potential high latency for operation op_mkdirs. latencyMs=153; previousMaxLatencyMs=0; operationCount=1; context=gs://dataproc-temp-us-east1-89254757867-nbcdhrwu/e4eba3e9-a7d5-490f-86ed-ed80b0f70b03/spark-job-history
24/04/29 00:55:02 INFO org.apache.hadoop.mapred.FileInputFormat: Total input files to process : 1
24/04/29 00:55:08 INFO com.google.cloud.hadoop.fs.gcs.GhfsStorageStatistics: Detected potential high latency for operation op_mkdirs. latencyMs=204; previousMaxLatencyMs=153; operationCount=2; context=gs://ssudhir2-stat606s24-hw11/WP_extremes/_temporary/0
24/04/29 00:55:14 INFO com.google.cloud.hadoop.repackaged.gcs.com.google.cloud.hadoop.gcsio.GoogleCloudStorageFileSystem: Successfully repaired 'gs://ssudhir2-stat606s24-hw11/WP_extremes/' directory.
24/04/29 00:55:14 INFO com.google.cloud.hadoop.fs.gcs.GhfsStorageStatistics: Detected potential high latency for operation op_delete. latencyMs=445; previousMaxLatencyMs=0; operationCount=1; context=gs://ssudhir2-stat606s24-hw11/WP_extremes/_temporary
24/04/29 00:55:14 INFO com.google.cloud.hadoop.fs.gcs.GhfsStorageStatistics: Detected potential high latency for operation stream_write_close_operations. latencyMs=195; previousMaxLatencyMs=0; operationCount=1; context=gs://ssudhir2-stat606s24-hw11/WP_extremes/_SUCCESS
24/04/29 00:55:14 INFO org.sparkproject.jetty.server.AbstractConnector: Stopped Spark@3b14e2b3{HTTP/1.1, (http/1.1)}{0.0.0.0:0}
24/04/29 00:55:15 INFO com.google.cloud.hadoop.fs.gcs.GhfsStorageStatistics: Detected potential high latency for operation op_rename. latencyMs=164; previousMaxLatencyMs=0; operationCount=1; context=rename(gs://dataproc-temp-us-east1-89254757867-nbcdhrwu/e4eba3e9-a7d5-490f-86ed-ed80b0f70b03/spark-job-history/application_1714341781891_0009.inprogress -> gs://dataproc-temp-us-east1-89254757867-nbcdhrwu/e4eba3e9-a7d5-490f-86ed-ed80b0f70b03/spark-job-history/application_1714341781891_0009)
```

Now, we check if the job was run successfully on our created cluster:

```bash
ssudhir2@ssudhir2-stat606s24-hw11-m:~$ gsutil ls  gs://ssudhir2-stat606s24-hw11/WP_extremes/
gs://ssudhir2-stat606s24-hw11/WP_extremes/
gs://ssudhir2-stat606s24-hw11/WP_extremes/_SUCCESS
gs://ssudhir2-stat606s24-hw11/WP_extremes/part-00000
gs://ssudhir2-stat606s24-hw11/WP_extremes/part-00001
gs://ssudhir2-stat606s24-hw11/WP_extremes/part-00002
gs://ssudhir2-stat606s24-hw11/WP_extremes/part-00003
```

Concatenating `part-00000`, `part-00001`, `part-00002`, and `part-00003` in our bash terminal as follows:

```bash
ssudhir2@ssudhir2-stat606s24-hw11-m:~$ gsutil cat gs://ssudhir2-stat606s24-hw11/WP_extremes/part-* > extremes.txt
ssudhir2@ssudhir2-stat606s24-hw11-m:~$ ls
avgs.txt  extremes.txt  ps_year_avgs.py  ps_year_extremes.py  wp_output.txt
```

Now, we transfer `extremes.txt` to the cloud bucket `gs://ssudhir2-stat606s24-hw11/` as follows:

```bash
ssudhir2@ssudhir2-stat606s24-hw11-m:~$ gsutil cp extremes.txt gs://ssudhir2-stat606s24-hw11/
Copying file://extremes.txt [Content-Type=text/plain]...
/ [1 files][  1.9 KiB/  1.9 KiB]                                                
Operation completed over 1 objects/1.9 KiB.                                      
```

Now, we check if our output is in the cloud directory as expected:

```bash
ssudhir2@ssudhir2-stat606s24-hw11-m:~$ gsutil ls gs://ssudhir2-stat606s24-hw11/
gs://ssudhir2-stat606s24-hw11/avgs.txt
gs://ssudhir2-stat606s24-hw11/extremes.txt
gs://ssudhir2-stat606s24-hw11/ps_year_avgs.py
gs://ssudhir2-stat606s24-hw11/ps_year_extremes.py
gs://ssudhir2-stat606s24-hw11/wp_output.txt
gs://ssudhir2-stat606s24-hw11/WP_averages/
gs://ssudhir2-stat606s24-hw11/WP_extremes/
gs://ssudhir2-stat606s24-hw11/WP_wordcount/
```

Finally, we check our running clusters, and delete them as required:

```bash
ssudhir2@cloudshell:~ (ssudhir2-stat606s24)$ gcloud dataproc clusters list --region=us-east1
NAME: ssudhir2-stat606s24-hw11
PLATFORM: GCE
PRIMARY_WORKER_COUNT: 2
SECONDARY_WORKER_COUNT: 
STATUS: RUNNING
ZONE: us-east1-d
SCHEDULED_DELETE: 
ssudhir2@cloudshell:~ (ssudhir2-stat606s24)$ gcloud dataproc clusters delete ssudhir2-stat606s24-hw11 --region=us-east1
The cluster 'ssudhir2-stat606s24-hw11' and all attached disks will be deleted.

Do you want to continue (Y/n)?  Y

Waiting on operation [projects/ssudhir2-stat606s24/regions/us-east1/operations/68b6ca5d-6422-3dcb-a817-b5b17b8d2d93].
Waiting for cluster deletion operation...done.                                                                                                                             
Deleted [https://dataproc.googleapis.com/v1/projects/ssudhir2-stat606s24/regions/us-east1/clusters/ssudhir2-stat606s24-hw11].
```